# `locityper_04_dispatch` — keep list, shards, retry

Run **in a VWB JupyterLab app** after
[`locityper_03_budget.ipynb`](locityper_03_budget.ipynb)
has written `keep.csv`.

Reads the keep list from the run ledger, skips samples that already have
GCS outputs, shards the rest into Batch jobs (TASKS_TSV + `BATCH_TASK_INDEX`,
under the 1 MiB job-JSON limit), polls, and resubmits FAILED / missing
outputs with a **new** job id. Never retries SUCCEEDED+objects unless
`FORCE_RERUN`.

`DISPATCH_*` stay **off** until you turn one on. Native Batch has no DAG:
minicram first, then genotype. `_02` remains the one-sample smoke. Status of
every job: [`batch_monitor.ipynb`](batch_monitor.ipynb).


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "dispatch.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "dispatch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch. Clone kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from vwb_batch import alloc, cohort, cost, dispatch, gcs, poll, registry  # noqa: E402
from vwb_batch.pipelines import locityper as lt  # noqa: E402

SCRATCH = Path.cwd() / "locityper_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

OUTPUT_BUCKET_GS = os.environ.get("LT_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", "us-central1")
RUN_ID = os.environ.get("LT_RUN_ID", "").strip() or "PASTE_FROM_03"
if RUN_ID in {"", "PASTE_FROM_03"}:
    raise SystemExit("set RUN_ID / LT_RUN_ID from locityper_03_budget")

OUT_PREFIX = os.environ.get(
    "LT_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/locityper",
)
PRINT_READS_DOCKER = os.environ.get(
    "LOCITYPER_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)
LOCITYPER_DOCKER = os.environ.get(
    "LOCITYPER_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/eichlerlab/locityper:1.4.5.0",
)
SHARD_SIZE = int(os.environ.get("LT_SHARD_SIZE", "200"))
PARALLELISM = int(os.environ.get("LT_DISPATCH_PARALLELISM", "200"))
MAX_RUN_DURATION = os.environ.get("LT_MAX_RUN_DURATION", "28800s")

DISPATCH_MINICRAM = False
DISPATCH_GENOTYPE = False
FORCE_RERUN = False
USE_TUNED_COMPUTE = True

PATHS = registry.run_paths(
    output_bucket=OUTPUT_BUCKET_GS.rstrip("/"),
    pipeline="locityper",
    run_id=RUN_ID,
)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_ID:", RUN_ID)
print("ledger:", PATHS.prefix)
print("keep.csv:", PATHS.keep_csv)
print("OUT_PREFIX:", OUT_PREFIX)
print("SHARD_SIZE:", SHARD_SIZE, "PARALLELISM:", PARALLELISM)
print("DISPATCH_MINICRAM:", DISPATCH_MINICRAM, "DISPATCH_GENOTYPE:", DISPATCH_GENOTYPE)
print("FORCE_RERUN:", FORCE_RERUN, "USE_TUNED_COMPUTE:", USE_TUNED_COMPUTE)
print("PET_SA_EMAIL:", os.environ.get("PET_SA_EMAIL", "(unset)"))
print("gcloud:", shutil.which("gcloud"))


## PET SA / VPC


In [ ]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
project, sa = alloc.require_vwb_env()
if not gcs.exists(PATHS.keep_csv):
    raise SystemExit(f"missing keep.csv: {PATHS.keep_csv} (run locityper_03_budget)")
print("project:", project)
print("PET_SA_EMAIL:", sa)
print("run.json exists:", gcs.exists(PATHS.run_json))


## Load keep list + current objects

One recursive listing of `OUT_PREFIX`, then classify each keep-list sample.
Does not submit.


In [ ]:
project, sa = alloc.require_vwb_env()
local_keep = SCRATCH / f"{RUN_ID}.keep.csv"
gcs.download_file(PATHS.keep_csv, local_keep)
keep_records = cohort.read_csv(local_keep)
print("keep:", len(keep_records))

run_meta = registry.load_run_json(PATHS)
COMPUTE = run_meta.get("compute") or {}
print("tuned compute:", COMPUTE or "(none — dispatch will use WDL defaults)")

listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
print("listed objects:", len(listing))
mini_pending, mini_done = poll.pending_records(
    keep_records,
    stage="minicram",
    out_prefix=OUT_PREFIX,
    expected=lt.expected_objects,
    force=FORCE_RERUN,
    listing=listing,
)
gt_pending, gt_done = poll.pending_records(
    mini_done if not FORCE_RERUN else keep_records,
    stage="genotype",
    out_prefix=OUT_PREFIX,
    expected=lt.expected_objects,
    force=FORCE_RERUN,
    listing=listing,
)
print("minicram done/pending:", len(mini_done), len(mini_pending))
print("genotype done/pending (among minicram-ready):", len(gt_done), len(gt_pending))
print("fully done:", len(gt_done))


## Dispatch minicram

Set `DISPATCH_MINICRAM = True`. Skips in-flight samples from the ledger.
Shards at `SHARD_SIZE` (job JSON stays tiny; workers read TASKS_TSV).
Uses `run.json` `compute.minicram` from the budget notebook when
`USE_TUNED_COMPUTE` is on.


In [ ]:
project, sa = alloc.require_vwb_env()
inflight = poll.inflight_sample_ids(registry.load_events(PATHS), "minicram")
todo = [r for r in mini_pending if r["sample_id"] not in inflight]
print("minicram todo:", len(todo), "in-flight:", len(inflight))
mini_compute = COMPUTE.get("minicram") if USE_TUNED_COMPUTE else None
print("minicram compute:", mini_compute or "default")
if not DISPATCH_MINICRAM:
    print("DISPATCH_MINICRAM is off; not submitting.")
elif not todo:
    print("nothing to submit")
else:
    n_existing = sum(1 for ev in registry.load_events(PATHS) if ev.get("event") == "job_submitted" and ev.get("stage") == "minicram")
    for i, (shard, chunk) in enumerate(cohort.shards(todo, SHARD_SIZE)):
        rows = lt.env_rows(chunk, stage="minicram", out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS,
            stage="minicram",
            shard=n_existing + shard,
            records=chunk,
            env_rows=rows,
            build_job=lt.build_job,
            image=PRINT_READS_DOCKER,
            out_prefix=OUT_PREFIX,
            project=project,
            region=REGION,
            sa=sa,
            parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION,
            scratch=SCRATCH,
            compute=mini_compute,
        )
        dispatch.maybe_submit(
            prepared=prepared,
            paths=PATHS,
            stage="minicram",
            shard=n_existing + shard,
            project=project,
            region=REGION,
            do_submit=True,
        )


## Poll all ledger jobs

Safe to re-run any time. Writes `status.tsv`. Does not submit.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
now = registry.utcnow()
status_rows = []
seen = set()
for ev in registry.load_events(PATHS):
    if ev.get("event") != "job_submitted":
        continue
    job_id = ev["job_id"]
    if job_id in seen:
        continue
    seen.add(job_id)
    stage = ev["stage"]
    try:
        polled = poll.poll_job(job_id=job_id, project=project, region=REGION)
    except RuntimeError as exc:
        print("skip", job_id, exc)
        continue
    registry.append_event(PATHS, "job_polled", stage=stage, job_id=job_id, state=polled["state"])
    print(job_id, stage, polled["state"], "tasks", len(polled["tasks"]))
    status_rows.extend(
        poll.reconcile_shard(
            sample_ids=ev.get("sample_ids") or [],
            stage=stage,
            job_id=job_id,
            shard=int(ev.get("shard") or 0),
            poll=polled,
            expected=lambda sid, st, stage=stage: lt.expected_objects(sid, st, OUT_PREFIX),
            now=now,
            listing=listing,
        )
    )
registry.write_status(PATHS, status_rows)
from collections import Counter
print(Counter((r.get("stage"), r.get("state")) for r in status_rows))
print("status:", PATHS.status_tsv)


## Dispatch genotype

Only keep-list samples with minicram objects. Set `DISPATCH_GENOTYPE = True`.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
mini_pending, mini_done = poll.pending_records(
    keep_records,
    stage="minicram",
    out_prefix=OUT_PREFIX,
    expected=lt.expected_objects,
    listing=listing,
)
gt_pending, gt_done = poll.pending_records(
    mini_done,
    stage="genotype",
    out_prefix=OUT_PREFIX,
    expected=lt.expected_objects,
    force=FORCE_RERUN,
    listing=listing,
)
inflight = poll.inflight_sample_ids(registry.load_events(PATHS), "genotype")
todo = [r for r in gt_pending if r["sample_id"] not in inflight]
print("minicram remaining:", len(mini_pending))
print("genotype todo:", len(todo), "done:", len(gt_done), "in-flight:", len(inflight))
gt_compute = COMPUTE.get("genotype") if USE_TUNED_COMPUTE else None
print("genotype compute:", gt_compute or "default")
if not DISPATCH_GENOTYPE:
    print("DISPATCH_GENOTYPE is off; not submitting.")
elif not todo:
    print("nothing to submit")
else:
    n_existing = sum(1 for ev in registry.load_events(PATHS) if ev.get("event") == "job_submitted" and ev.get("stage") == "genotype")
    for shard, chunk in cohort.shards(todo, SHARD_SIZE):
        rows = lt.env_rows(chunk, stage="genotype", out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS,
            stage="genotype",
            shard=n_existing + shard,
            records=chunk,
            env_rows=rows,
            build_job=lt.build_job,
            image=LOCITYPER_DOCKER,
            out_prefix=OUT_PREFIX,
            project=project,
            region=REGION,
            sa=sa,
            parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION,
            scratch=SCRATCH,
            compute=gt_compute,
        )
        dispatch.maybe_submit(
            prepared=prepared,
            paths=PATHS,
            stage="genotype",
            shard=n_existing + shard,
            project=project,
            region=REGION,
            do_submit=True,
        )


## Retry missing / failed

Re-run the dispatch cells after poll. `pending_records` already excludes
SUCCEEDED+objects. `FORCE_RERUN` is the only way to redo those.

Cost rollup below uses completed keep-list samples (same rates as the budget
notebook). Billing export still lags.


In [ ]:
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
status = {(r.get("sample_id"), r.get("stage")): r for r in registry.load_status(PATHS)}
costs = []
for rec in keep_records:
    sid = rec["sample_id"]
    if not all(u in listing for u in lt.expected_objects(sid, "minicram", OUT_PREFIX)):
        continue
    if not all(u in listing for u in lt.expected_objects(sid, "genotype", OUT_PREFIX)):
        continue
    try:
        network = cost.parse_transfer_stats(gcs.cat(lt.transfer_stats_uri(sid, OUT_PREFIX)))
    except Exception:
        continue
    row = cost.sample_cost_usd(
        minicram_seconds=float(status.get((sid, "minicram"), {}).get("seconds") or 0) or 0.0,
        genotype_seconds=float(status.get((sid, "genotype"), {}).get("seconds") or 0) or 0.0,
        network_bytes=network,
    )
    costs.append(row)
print("fully done on keep list:", len(costs), "/", len(keep_records))
if costs:
    s = cost.summarize_pilot(costs, q=0.90)
    print("mean", round(s["mean_usd"], 4), "p90", round(s["p90_usd"], 4), "p95", round(s["p95_usd"], 4))
    print("sum of completed (estimate):", round(sum(c["total_usd"] for c in costs), 2))
